### Stream Customers Data from cloud files to Delta Lake

 1. Read files from cloud storage using DataStreamReader API
 2. Transform the dataframe to add the following columns\
        - file path: Cloud file path\
        - ingestion date: Current Time Stamp
 3. Write the transformed data stream to a Delta lake table

**1. Read files from Cloud storage using DataStreamReader API**

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType

customers_schema = StructType(fields =[
                            StructField("customer_id", IntegerType()),
                            StructField("customer_name", StringType()),
                            StructField("date_of_birth", DateType()),
                            StructField("telephone", StringType()),
                            StructField("email", StringType()),
                            StructField("member_since", DateType()),
                            StructField("created_timestamp", TimestampType()),
])

In [0]:
customers_df = (
    spark.readStream
        .format("json")
        .schema(customers_schema)        
        .load("/Volumes/gizmobox_sankara/landing/operational_volume/customers_stream/")
      )

**2. Transform the dataframe to add the following columns**
- file path: Cloud file path
- ingestion data: current time stamp

In [0]:
from pyspark.sql.functions import col, current_timestamp

customers_transformed_df = (
    customers_df.withColumn("file_path", col("_metadata.file_path"))
                .withColumn("ingestion_date", current_timestamp())
)

**3. Write the transformed data stream to a Delta lake table**

[structured-streaming-output-mode](https://learn.microsoft.com/en-us/azure/databricks/structured-streaming/output-mode)

[structured-streaming-triggers](https://learn.microsoft.com/en-us/azure/databricks/structured-streaming/triggers)

In [0]:
streaming_query = (
    customers_transformed_df.writeStream
    .format("delta")
    .option("checkpointlocation", "/Volumes/gizmobox_sankara/landing/operational_volume/customers_stream/_checkpoint_stream")
    .toTable("gizmobox_sankara.bronze.customers_stream")
)


In [0]:
streaming_query.stop()

In [0]:
spark.sql("drop table if exists gizmobox_sankara.bronze.customers_stream")

In [0]:
spark.table("gizmobox_sankara.bronze.customers_stream").display()